In [2]:
import pandas as pd 
import numpy as np  

In [3]:
orders = pd.read_csv(r'C:\Users\Ayoub-PC\Desktop\Olist Analysis Project\Data\Raw\olist_orders_dataset.csv')
customers = pd.read_csv(r'C:\Users\Ayoub-PC\Desktop\Olist Analysis Project\Data\Raw\olist_customers_dataset.csv')   
products = pd.read_csv(r'C:\Users\Ayoub-PC\Desktop\Olist Analysis Project\Data\Raw\olist_products_dataset.csv')
order_items = pd.read_csv(r'C:\Users\Ayoub-PC\Desktop\Olist Analysis Project\Data\Raw\olist_order_items_dataset.csv')
sellers = pd.read_csv(r'C:\Users\Ayoub-PC\Desktop\Olist Analysis Project\Data\Raw\olist_sellers_dataset.csv')
order_payments = pd.read_csv(r'C:\Users\Ayoub-PC\Desktop\Olist Analysis Project\Data\Raw\olist_order_payments_dataset.csv')
order_reviews = pd.read_csv(r'C:\Users\Ayoub-PC\Desktop\Olist Analysis Project\Data\Raw\olist_order_reviews_dataset.csv')  
geolocation = pd.read_csv(r'C:\Users\Ayoub-PC\Desktop\Olist Analysis Project\Data\Raw\olist_geolocation_dataset.csv')  
Translation = pd.read_csv(r'C:\Users\Ayoub-PC\Desktop\Olist Analysis Project\Data\Raw\product_category_name_translation.csv')  

## Orders Dataset 

In [4]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [5]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


In [6]:
orders.describe().T

,count,unique,top,freq
order_id,99441,99441,e481f51cbdc54678b7cc49136f2d6af7,1
customer_id,99441,99441,9ef432eb6251297304e76186b10a928d,1
order_status,99441,8,delivered,96478
order_purchase_timestamp,99441,98875,2018-04-11 10:48:14,3
order_approved_at,99281,90733,2018-02-27 04:31:10,9
order_delivered_carrier_date,97658,81018,2018-05-09 15:48:00,47
order_delivered_customer_date,96476,95664,2018-05-08 23:38:46,3
order_estimated_delivery_date,99441,459,2017-12-20 00:00:00,522


In [7]:
#check missing values
orders.isna().sum()


order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [8]:
#check duplicates
orders.duplicated().sum()

0

In [9]:
#Orphan check
order_items_without_orders = (
    order_items
    .loc[~order_items['order_id'].isin(orders['order_id']), 'order_id']
    .nunique()
)

order_items_without_orders  # should be 0

0

In [10]:
# reverse check
orders_without_items = (
    orders
    .loc[~orders['order_id'].isin(order_items['order_id']), 'order_id']
    .nunique()
)

orders_without_items  # understand if > 0


775

In [11]:
#Join cardinality sanity check (MOST IMPORTANT)
orders_count = orders['order_id'].nunique()

joined_count = (
    orders
    .merge(order_items, on='order_id', how='left')
    ['order_id']
    .nunique()
)

orders_count, joined_count


(99441, 99441)

In [12]:
#convert date columns to datetime
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_approved_at'] = pd.to_datetime(orders['order_approved_at'])
orders['order_delivered_carrier_date'] = pd.to_datetime(orders['order_delivered_carrier_date'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])
orders['order_estimated_delivery_date'] = pd.to_datetime(orders['order_estimated_delivery_date'])   

### Orders Dataset notes
- Some orders didn't get through the sales pipeline (We should analyze why)
- There are 775 orders with no items 
- We have datetime columns that are objects (It is recommended to change the data type)

## Order Items dataset

In [13]:
order_items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  object 
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  object 
 3   seller_id            112650 non-null  object 
 4   shipping_limit_date  112650 non-null  object 
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.0+ MB


In [14]:
order_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [15]:
order_items.isnull().sum()  

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

In [16]:
order_items.duplicated().sum()  

0

In [17]:
order_items.duplicated(subset=['order_id']).sum()

13984

In [18]:
#orphan check
missing_products = (   
    products
    .loc[~products['product_id'].isin(order_items['product_id']), 'product_id']
    .nunique()
)
missing_products  # should be 0

missing_sellers = (
    sellers
    .loc[~sellers['seller_id'].isin(order_items['seller_id']), 'seller_id']
    .nunique()
)
missing_sellers  # should be 0

0

In [19]:
#reverse orphan check
products_no_items = (
    order_items
    .loc[~order_items['product_id'].isin(products['product_id']), 'product_id']
    .nunique()
)
products_no_items

sellers_no_items = (
    order_items
    .loc[~order_items['seller_id'].isin(sellers['seller_id']), 'seller_id']
    .nunique()
)
sellers_no_items

0

In [20]:
#join cardinality sanity check for products and sellers
order_items_count = order_items['order_item_id'].nunique()
joined_count = (
    order_items
    .merge(products, on='product_id', how='left')
    ['order_item_id']
    .nunique()
)
order_items_count, joined_count

joined_count = (
    order_items
    .merge(sellers, on='seller_id', how='left')
    ['order_item_id']   
    .nunique()
)
order_items_count, joined_count

(21, 21)

In [21]:
#convert date columns to datetime  
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp']) 

### Order Items Dataset notes
- There are 13984 duplicated order_id in order Items which makes sense because order item id identifies the number of items included in the same order and each sequence is in one row with the same order_id. 
- Converted columns to datetime

## Customers dataset

In [22]:
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [23]:
customers.isnull().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

In [24]:
customers.duplicated(subset=['customer_unique_id']).sum()    

3345

In [25]:
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB


### Customers dataset notes
- 3345 duplicated customer unique ID which can be interpreted for repeated customers

## Order Payment dataset 

In [26]:
order_payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [27]:
order_payments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  object 
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  object 
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), object(2)
memory usage: 4.0+ MB


In [28]:
order_payments.isnull().sum()

order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

In [29]:
order_payments.duplicated(subset=['order_id']).sum()   

4446

In [30]:
#orphan check
missing_orders = (   
    order_payments
    .loc[~order_payments['order_id'].isin(orders['order_id']), 'order_id']
    .nunique()
)
missing_orders  # should be 0

0

In [31]:
#reverse orphan check
payments_no_orders = (   
    orders
    .loc[~orders['order_id'].isin(order_payments['order_id']), 'order_id']
    .nunique()
)
payments_no_orders

1

In [32]:
#join cardinality sanity check
orders_count = orders['order_id'].nunique()
joined_count = (
    orders
    .merge(order_payments, on='order_id', how='right')
    ['order_id']
    .nunique()
)
orders_count, joined_count


(99441, 99440)

### Order Payment dataset notes 
- There are 4446 duplicated order ids ? meaning the same order has payed more than once ? 
- there is 1 order with no payment ?

## Sellers dataset 

In [33]:
sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [34]:
sellers.isnull().sum()  

seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

In [35]:
sellers.duplicated().sum()

0

In [36]:
sellers.duplicated(subset=['seller_id']).sum()

0

In [37]:
#orphan check
missing_sellers = (   
    order_items
    .loc[~order_items['seller_id'].isin(sellers['seller_id']), 'seller_id']
    .nunique()
)
missing_sellers  # should be 0

0

In [38]:
#reverse orphan check
sellers_no_orders = (   
    sellers
    .loc[~sellers['seller_id'].isin(order_items['seller_id']), 'seller_id']
    .nunique()
)
sellers_no_orders

0

In [39]:
#join cardinality sanity check
order_items_count = order_items['seller_id'].nunique()
joined_count = (
    order_items
    .merge(sellers, on='seller_id', how='left')
    ['seller_id']
    .nunique()
)
order_items_count, joined_count 

(3095, 3095)

In [40]:
#reverse join cardinality sanity check
sellers_count = sellers['seller_id'].nunique()
joined_count = (
    sellers.merge(order_items, on='seller_id', how='right')
    ['seller_id']
    .nunique()
)
sellers_count, joined_count

(3095, 3095)

### Sellers dataset notes 
We're all good here

## Order Reviews Dataset 

In [41]:
order_reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [42]:
order_reviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   review_id                99224 non-null  object
 1   order_id                 99224 non-null  object
 2   review_score             99224 non-null  int64 
 3   review_comment_title     11568 non-null  object
 4   review_comment_message   40977 non-null  object
 5   review_creation_date     99224 non-null  object
 6   review_answer_timestamp  99224 non-null  object
dtypes: int64(1), object(6)
memory usage: 5.3+ MB


In [43]:
order_reviews.isnull().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [44]:
order_reviews.duplicated().sum()    

0

In [45]:
order_reviews.duplicated(subset=['order_id'], keep=False)

0        False
1        False
2        False
3        False
4        False
         ...  
99219    False
99220    False
99221    False
99222    False
99223    False
Length: 99224, dtype: bool

In [46]:
#orphan check
orders_without_order_reviews = (
    orders
    .loc[~orders['order_id'].isin(order_reviews['order_id']), 'order_id']
    .nunique()
)
orders_without_order_reviews  # understand if > 0

768

In [47]:
#reverse orphan check
order_reviews_without_orders = (
    order_reviews.loc[~order_reviews['order_id'].isin(orders['order_id']), 'order_id'].nunique()
)
order_reviews_without_orders  # should be 0

0

In [48]:
#join cardinality sanity check
order_reviews_count = order_reviews['order_id'].nunique()
joined_count = (
    orders
    .merge(order_reviews, on='order_id', how='right')
    ['order_id']
    .nunique()
)
order_reviews_count, joined_count

(98673, 98673)

In [49]:
#convert date columns to datetime
order_reviews['review_creation_date'] = pd.to_datetime(order_reviews['review_creation_date'])
order_reviews['review_answer_timestamp'] = pd.to_datetime(order_reviews['review_answer_timestamp'])

### Order Reviews dataset notes 
- Change columns to datetime 
- Missing 87656 review comment title 
- Missing 58247 review comment message 

which makes sense because not everyone leaves a review message (We have to figure out what to do with the missing data)
- 551 order id duplicated which means the same order id has reviewed multiple times. 
- there are 768 orders without order reviews

## Products Dataset 

In [50]:
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [51]:
products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  object 
 1   product_category_name       32341 non-null  object 
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), object(2)
memory usage: 2.3+ MB


In [52]:
products.isnull().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [53]:
products.duplicated(subset=['product_id'])

0        False
1        False
2        False
3        False
4        False
         ...  
32946    False
32947    False
32948    False
32949    False
32950    False
Length: 32951, dtype: bool

In [54]:
#orphan check
products_without_order_items = (
    products.loc[~products['product_id'].isin(order_items['product_id']), 'product_id']
).nunique()
products_without_order_items  # should be 0

0

In [55]:
#reverse orphan check
order_items_without_products = (
    order_items.loc[~order_items['product_id'].isin(products['product_id']), 'product_id']
).nunique()
order_items_without_products  # should be 0

0

In [56]:
#join cardinality sanity check
products_count = products['product_id'].nunique()
joined_count = (
    products
    .merge(order_items, on='product_id', how='left')
    ['product_id']
    .nunique()
)
products_count, joined_count

(32951, 32951)

### Products dataset notes 
- 610 missing : product_category_name, product_name_lenght, product_description_lenght, product_photos_qty
- 2 missing : product_weight_g, product_length_cm, product_height_cm, product_width_cm 
- We need to figure out how to deal with the missing data if we are going to use it. 

## Geolocation Dataset 

In [57]:
geolocation.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


In [58]:
geolocation.isnull().sum()

geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

In [59]:
geolocation.duplicated()

0          False
1          False
2          False
3          False
4          False
           ...  
1000158    False
1000159     True
1000160     True
1000161    False
1000162     True
Length: 1000163, dtype: bool

In [60]:
geolocation.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  object 
 4   geolocation_state            1000163 non-null  object 
dtypes: float64(2), int64(1), object(2)
memory usage: 38.2+ MB


## Translation Dataset  

In [61]:
Translation.head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [63]:
#saving the cleaned datasets in processed folder
orders.to_csv(r'C:\Users\Ayoub-PC\Desktop\Olist Analysis Project\Data\Processed\orders_cleaned.csv', index=False)
customers.to_csv(r'C:\Users\Ayoub-PC\Desktop\Olist Analysis Project\Data\Processed\customers_cleaned.csv', index=False)
products.to_csv(r'C:\Users\Ayoub-PC\Desktop\Olist Analysis Project\Data\Processed\products_cleaned.csv', index=False)
order_items.to_csv(r'C:\Users\Ayoub-PC\Desktop\Olist Analysis Project\Data\Processed\order_items_cleaned.csv', index=False)
sellers.to_csv(r'C:\Users\Ayoub-PC\Desktop\Olist Analysis Project\Data\Processed\sellers_cleaned.csv', index=False)
order_payments.to_csv(r'C:\Users\Ayoub-PC\Desktop\Olist Analysis Project\Data\Processed\order_payments_cleaned.csv', index=False)
order_reviews.to_csv(r'C:\Users\Ayoub-PC\Desktop\Olist Analysis Project\Data\Processed\order_reviews_cleaned.csv', index=False) 
geolocation.to_csv(r'C:\Users\Ayoub-PC\Desktop\Olist Analysis Project\Data\Processed\geolocations_cleaned.csv', index=False)
Translation.to_csv(r'C:\Users\Ayoub-PC\Desktop\Olist Analysis Project\Data\Processed\translation_cleaned.csv', index=False)
